# Aurora Forecasts - Monthly Processing

## Overview

This notebook processes monthly Aurora forecast data and builds weighted capture price matrices aggregating generation data across technology subgroups.

### Key Functions:
- Processes monthly-resolution forecasts
- Calculates weighted average capture prices
- Weights by generation output
- Analyzes renewable generation economics
- Enables month-level time series analysis


## 1. Import Dependencies

Import required libraries for monthly data processing.

In [9]:
import pandas as pd
from typing import Tuple

In [10]:
technology_path_monthly = '../../../data/aurora/aurora_technology_scenarios_ES_default_currency_1m.parquet'
system_path_monthly     = '../../../data/aurora/aurora_system_scenarios_ES_default_currency_1m.parquet'
technology_path_yearly = '../../../data/aurora/aurora2_technology_scenarios_ES_default_currency_1y.parquet'

inflation_path = '../../../data/finance/inflation.xlsx'

In [11]:
def build_weighted_capture_matrices_aurora(
    df: pd.DataFrame,
    price_col: str = "Uncurtailed capture price",
    gen_col: str = "Generation",
    group_col: str = "Group",
    by: Tuple[str, ...] = ("name", "year"),   # index of the matrix
):
    """
    Returns three DataFrames:
      1) long_df  : per (by..., Group) with total generation and weighted capture price
      2) price_w  : wide matrix of weighted capture prices by Group
      3) gen_w    : wide matrix of total generation by Group

    Weighted capture price = sum(price * generation) / sum(generation)
    """
    df = df.copy()

    # Make sure the numeric columns are numeric
    df[gen_col] = pd.to_numeric(df[gen_col], errors="coerce")
    df[price_col] = pd.to_numeric(df[price_col], errors="coerce")

    # Drop rows without data to avoid NaNs in weights
    df = df.dropna(subset=[gen_col, price_col])

    # Aggregate over subgroups: compute generation sum and weighted price per (by..., Group)
    def _wavg(g):
        gsum = g[gen_col].sum()
        return pd.Series({
            "Generation": gsum,
            "Weighted capture price": (g[price_col].mul(g[gen_col])).sum() / gsum if gsum else float("nan")
        })

    long_df = (
        df.groupby(list(by) + [group_col], dropna=False)
          .apply(_wavg)
          .reset_index()
    )

    # Build the matrices
    price_w = long_df.pivot_table(
        index=list(by), columns=group_col, values="Weighted capture price"
    ).sort_index()
    gen_w = long_df.pivot_table(
        index=list(by), columns=group_col, values="Generation"
    ).sort_index()

    # Clean column names for nicer display
    price_w.columns.name = None
    gen_w.columns.name = None

    return long_df, price_w, gen_w


In [12]:
inflation_figures_df = pd.read_excel(inflation_path, sheet_name='Sheet1')

for column in range(2015,2063):
    inflation_figures_df[column] = inflation_figures_df[column].astype(float)
    
inflation_figures_df.iloc[2, 5]
inflation_figures_df

,Date,Country,Type,1980,1981,1982,1983,1984,1985,1986,...,2053,2054,2055,2056,2057,2058,2059,2060,2061,2062
0,2025-Apr,France,Inflation (annual % change),13.1,13.3,12.0,9.5,7.7,5.8,2.5,...,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.9,1.9
1,2025-Apr,Germany,Inflation (annual % change),5.4,6.3,5.3,3.3,2.4,2.1,-0.1,...,2.2,2.2,2.2,2.2,2.2,2.2,2.2,2.2,2.2,2.2
2,2025-Apr,Ireland,Inflation (annual % change),18.3,20.2,17.2,10.4,8.6,5.5,3.0,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0
3,2025-Apr,Italy,Inflation (annual % change),21.8,19.5,16.5,14.7,10.7,9.0,5.8,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0
4,2025-Apr,Poland,Inflation (annual % change),9.4,21.2,100.8,22.1,75.6,15.1,17.8,...,2.5,2.5,2.5,2.5,2.5,2.5,2.5,2.5,2.5,2.5
5,2025-Apr,Portugal,Inflation (annual % change),5.9,21.2,22.7,25.1,29.3,19.3,11.7,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0
6,2025-Apr,Spain,Inflation (annual % change),15.6,14.5,14.4,12.2,11.3,8.8,8.8,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0
7,2025-Apr,GB,Inflation (annual % change),16.8,12.2,8.5,5.2,4.4,5.2,3.6,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0
8,2025-Apr,World,Inflation (annual % change),17.3,15.0,14.3,13.4,14.0,13.7,11.7,...,3.2,3.2,3.2,3.2,3.2,3.2,3.2,3.2,3.2,3.2
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
df_technology_month = pd.read_parquet(technology_path_monthly)
df_system_month = pd.read_parquet(system_path_monthly)
df_technology_year = pd.read_parquet(technology_path_yearly)

In [14]:
# df_technology_month
df_system_month

,year,month,variable,value,units,name,currency,sensitivity,region_code
0,2022,<NA>,Carbon price,73.45,GBP/tonne,Great Britain Apr 2022 (Low),gbp2021,low,gbr
1,2023,<NA>,Carbon price,69.10,GBP/tonne,Great Britain Apr 2022 (Low),gbp2021,low,gbr
2,2024,<NA>,Carbon price,65.50,GBP/tonne,Great Britain Apr 2022 (Low),gbp2021,low,gbr
3,2025,<NA>,Carbon price,61.68,GBP/tonne,Great Britain Apr 2022 (Low),gbp2021,low,gbr
4,2026,<NA>,Carbon price,58.72,GBP/tonne,Great Britain Apr 2022 (Low),gbp2021,low,gbr
...,...,...,...,...,...,...,...,...,...
518310,2060,<NA>,Gas price,63.39,EUR/MWh,Iberia Q1 26 (High),eur2024,high,esp
518311,2060,<NA>,Gas price,63.00,EUR/MWh,Iberia Q1 26 (High),eur2024,high,esp
518312,2060,<NA>,Gas price,62.96,EUR/MWh,Iberia Q1 26 (High),eur2024,high,esp
518313,2060,<NA>,Gas price,61.90,EUR/MWh,Iberia Q1 26 (High),eur2024,high,esp


In [15]:
df_technology_month_fil = df_technology_month[
    (df_technology_month['type'].isin(['Uncurtailed capture price', 'Generation']))
    & 
    (df_technology_month['Group'].isin(['Solar', 'Onshore wind', 'Offshore wind']))
    ]

In [16]:
# As we do not have monthly generation data, we are going to import the yearly generation data and assume it is evenly distributed across months.
# We will add the yearly generation data to the monthly dataframe.

from utils.utils_dates import map_month

df_yearly_generation = df_technology_year[
    (df_technology_year['type'] == 'Generation') & (df_technology_year['Group'].isin(df_technology_month['Group'].unique()))
]

# We now have the yearly generation data for the target groups (Solar and Onshore wind). 
# For including the data in the monthly dataframe, we are going to have to replicate each yearly generation value 12 times (once per month) and then concatenate it with the monthly dataframe.
# This way we will be able to use the build_weighted_capture_matrices_aurora function to build the weighted capture price matrices.

df_yearly_generation = df_yearly_generation.copy()
df_yearly_generation['month'] = 1  # Start with January
df_yearly_generation = df_yearly_generation.loc[df_yearly_generation.index.repeat(12)]
df_yearly_generation['month'] = df_yearly_generation['month'] + df_yearly_generation.groupby(['name', 'year', 'Group', 'Subgroup', 'type']).cumcount()
df_yearly_generation['month'] = df_yearly_generation['month'].astype(int)
df_yearly_generation
df_yearly_generation['month'] = df_yearly_generation['month'].apply(map_month)

#We now adjust arrangement to match df_technology_month_fil and pivot type columns
df_yearly_generation = df_yearly_generation[df_technology_month_fil.columns]
df_technology_month_w_gen = pd.concat([df_technology_month_fil, df_yearly_generation], ignore_index=True)
df_technology_month_w_gen['type'].unique()

df_technology_month_w_gen_pivot = df_technology_month_w_gen.pivot(index=['name', 'year', 'month', 'Group', 'Subgroup', 'region_code', 'sensitivity', 'currency'], columns='type', values='value').reset_index()
df_technology_month_w_gen_pivot.dropna(subset=['Uncurtailed capture price', 'Generation'], inplace=True)
df_technology_month_w_gen_pivot['Group'].unique()


ModuleNotFoundError: No module named 'utils'

In [ ]:
long_df_all_month, price_by_name_month, gen_by_name_month = build_weighted_capture_matrices_aurora(df_technology_month_w_gen_pivot, by=("name", "year", "month", "currency", "region_code", "sensitivity"))

price_by_name_month.reset_index(inplace=True)

price_by_name_month = price_by_name_month.melt(id_vars=['name', 'year', 'month', 'currency', 'sensitivity'], var_name='Group', value_name='value')
long_df_all_month.drop(columns='Generation', inplace=True)

long_df_all_month

C:\Users\mpy\AppData\Local\Temp\ipykernel_21488\2854048230.py:35: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_wavg)


,name,year,month,currency,region_code,sensitivity,Group,Weighted capture price
0,Iberia Jan 24 (Low - No Generation Tax),2024,April,eur2022,esp,nogenerationtax,Onshore wind,32.21
1,Iberia Jan 24 (Low - No Generation Tax),2024,April,eur2022,esp,nogenerationtax,Solar,17.07
2,Iberia Jan 24 (Low - No Generation Tax),2024,August,eur2022,esp,nogenerationtax,Onshore wind,82.78
3,Iberia Jan 24 (Low - No Generation Tax),2024,August,eur2022,esp,nogenerationtax,Solar,61.40
4,Iberia Jan 24 (Low - No Generation Tax),2024,December,eur2022,esp,nogenerationtax,Onshore wind,58.01
...,...,...,...,...,...,...,...,...
59029,Iberia Oct 24 (Net Zero),2060,October,eur2023,esp,netzero,Onshore wind,76.11
59030,Iberia Oct 24 (Net Zero),2060,October,eur2023,esp,netzero,Solar,58.84
59031,Iberia Oct 24 (Net Zero),2060,September,eur2023,esp,netzero,Offshore wind,72.63
59032,Iberia Oct 24 (Net Zero),2060,September,eur2023,esp,netzero,Onshore wind,73.42


In [ ]:
def get_currency_change_by_annual_rates(
    value: float,
    inflation_map: dict,   # {year: percent}, e.g., {2023: 3.0, 2024: 2.2, 2025: 2.0}
    target_month: int,     # 1..12 (Jan..Dec)
    target_year: int = 2025,
    base_year: int = 2023,
):
    """
    Convert a value quoted in prices of DECEMBER of base_year into prices of (target_year, target_month),
    using Dec→Dec annual inflation rates (in PERCENT, e.g., 2 for 2%) and assuming a constant monthly rate
    within each year.

    Interpretation:
      - inflation_map[y] is the percent change from Dec(y-1) to Dec(y).
      - Within year y, the monthly factor is m_y = (1 + r_y) ** (1/12), where r_y = inflation_map[y]/100.
      - Reference date is Dec(base_year). The function inflates (forward) or deflates (backward) accordingly.

    Examples:
      - To move from Dec 2023 to Apr 2025 (target_year=2025, target_month=4), multiply by:
          (1+r_2024) * (1+r_2025)^(4/12)
      - To move from Dec 2023 back to May 2022, divide by the forward factor from May 2022 to Dec 2023:
          (1+r_2022)^((12-5)/12) * (1+r_2023)
    """
    if not (1 <= target_month <= 12):
        raise ValueError("target_month must be in 1..12")

    def r(y: int) -> float:
        if y not in inflation_map:
            raise KeyError(f"Missing annual rate (percent) for year {y}")
        return float(inflation_map[y]) / 100.0

    # Forward: from Dec(base_year) -> (target_year, target_month)
    if target_year > base_year:
        factor = 1.0
        # full years strictly between base_year and target_year
        for y in range(base_year + 1, target_year):
            factor *= (1.0 + r(y))
        # partial year within target_year: months 1..target_month
        factor *= (1.0 + r(target_year)) ** (target_month / 12.0)
        return value * factor

    # Same year: reference is Dec(base_year); we keep value as-is
    if target_year == base_year:
        # If you ever need months within the same base_year, consider adding a base_month param.
        return value

    # Backward: from Dec(base_year) -> (target_year, target_month) earlier in time
    # Compute the forward factor from (target_year, target_month) to Dec(base_year), then divide.
    factor_forward = 1.0
    # remaining months in target_year from target_month to Dec(target_year)
    factor_forward *= (1.0 + r(target_year)) ** ((12 - target_month) / 12.0)
    # full years strictly between target_year and base_year
    for y in range(target_year + 1, base_year):
        factor_forward *= (1.0 + r(y))
    # full base_year up to its December
    factor_forward *= (1.0 + r(base_year))
    return value / factor_forward


In [ ]:
def transform_to_real_prices_month(
        df: pd.DataFrame, 
        inflation_df: pd.DataFrame, 
        base_year: int, 
        country: str,
        year_col: str = 'year',
        month_col: str = 'month',
        price_cols: list = ['solar_capture', 'onshore_wind_capture'],
        ) -> pd.DataFrame:
    """
    Transforms nominal prices to real prices based on inflation data.

    Args:
        df (pd.DataFrame): DataFrame containing the prices to be transformed.
        inflation_df (pd.DataFrame): DataFrame containing inflation data with 'year' and 'inflation_factor' columns.
        base_year (int): The year to which prices should be adjusted (e.g., 2024).
        year_col (str): The name of the column in df that contains the year information.
        price_cols (list): List of column names in df that contain the prices to be transformed.

    Returns:
        pd.DataFrame: DataFrame with transformed real prices.
    """
    df = df.copy()
    df['currency_year'] = df['currency'].str[-4:].astype(int)

    # We are going to transform all prices to nominal prices in 2025, based on inflation_figures_df. First we are going to transform all prices to 
    # real 2024 prices, and the transform all prices to respective year forcasted prices. The transformation will be done using latest inflation data available.

    # Step 1: Transform all prices to real 2024 prices.

    inflation_df_fil2 = inflation_df[
        (inflation_df['Country'] == country)
        &
        (inflation_df['Date'] == '2025-Apr')       # este filtro es provisional, luego habría que ver como coger el ultimo dato disponible
        &
        (inflation_df['Type'] == 'Inflation (annual % change)') 
    ]

    inflation_map_nominal = inflation_df_fil2.iloc[0].to_dict()
    print(inflation_map_nominal)

    def _apply_row(row, price_col):
        return get_currency_change_by_annual_rates(
            value=row[price_col],
            inflation_map=inflation_map_nominal,
            target_month=int(row[month_col]),
            target_year=int(row[year_col]),
            base_year=int(row["currency_year"]),
        )

    for value in price_cols:
        df.loc[:, value] = df[value].astype(float)

        df.loc[:, f'nominal_{value}'] = df.apply(_apply_row, axis=1, args=(value,))

    return df

In [ ]:
# we are now going to transform prices to nominal prices, firstly changing all prices to real 2024 prices, and then transforming them to nominal prices for the respective year.

from utils.utils_dates import unmap_month

long_df_all_month2 = long_df_all_month.copy()

long_df_all_month2['month'] = long_df_all_month2['month'].apply(unmap_month)

df_technology_month_currency_transformed = transform_to_real_prices_month(
    df=long_df_all_month2,
    inflation_df=inflation_figures_df,
    base_year=2025,
    country='Spain',
    year_col='year',
    month_col='month',
    price_cols=['Weighted capture price']
)

df_technology_month_currency_transformed.to_pickle('c:/Users/mpy/Market data retrieval/api_retrieval/data/aurora/processed/aurora_technology_scenarios_ES_default_currency_1m_nominal_prices.pkl')

long_df_all_month2['year'].unique()

{'Date': '2025-Apr', 'Country': 'Spain', 'Type': 'Inflation (annual % change)', 2015: -0.6, 2016: -0.3, 2017: 2.0, 2018: 1.7, 2019: 0.8, 2020: -0.3, 2021: 3.0, 2022: 8.3, 2023: 3.4, 2024: 2.9, 2025: 2.2, 2026: 2.0, 2027: 2.1, 2028: 2.0, 2029: 2.0, 2030: 2.0, 2031: 2.0, 2032: 2.0, 2033: 2.0, 2034: 2.0, 2035: 2.0, 2036: 2.0, 2037: 2.0, 2038: 2.0, 2039: 2.0, 2040: 2.0, 2041: 2.0, 2042: 2.0, 2043: 2.0, 2044: 2.0, 2045: 2.0, 2046: 2.0, 2047: 2.0, 2048: 2.0, 2049: 2.0, 2050: 2.0, 2051: 2.0, 2052: 2.0, 2053: 2.0, 2054: 2.0, 2055: 2.0, 2056: 2.0, 2057: 2.0, 2058: 2.0, 2059: 2.0, 2060: 2.0, 2061: 2.0, 2062: 2.0}


array([2024, 2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034,
       2035, 2036, 2037, 2038, 2039, 2040, 2041, 2042, 2043, 2044, 2045,
       2046, 2047, 2048, 2049, 2050, 2051, 2052, 2053, 2054, 2055, 2056,
       2057, 2058, 2059, 2060, 2023, 2022])

In [ ]:
df_system_month_n = df_system_month.copy()
df_system_month_n['month'] = df_system_month_n['month'].apply(unmap_month)
df_system_month_currency_transformed = transform_to_real_prices_month(
    df=df_system_month_n,
    inflation_df=inflation_figures_df,
    base_year=2025,
    country='Spain',
    year_col='year',
    month_col='month',
    price_cols=['value']
)

df_system_month_currency_transformed.to_pickle('c:/Users/mpy/Market data retrieval/api_retrieval/data/aurora/processed/aurora_system_scenarios_ES_default_currency_1m_nominal_prices.pkl')


{'Date': '2025-Apr', 'Country': 'Spain', 'Type': 'Inflation (annual % change)', 2015: -0.6, 2016: -0.3, 2017: 2.0, 2018: 1.7, 2019: 0.8, 2020: -0.3, 2021: 3.0, 2022: 8.3, 2023: 3.4, 2024: 2.9, 2025: 2.2, 2026: 2.0, 2027: 2.1, 2028: 2.0, 2029: 2.0, 2030: 2.0, 2031: 2.0, 2032: 2.0, 2033: 2.0, 2034: 2.0, 2035: 2.0, 2036: 2.0, 2037: 2.0, 2038: 2.0, 2039: 2.0, 2040: 2.0, 2041: 2.0, 2042: 2.0, 2043: 2.0, 2044: 2.0, 2045: 2.0, 2046: 2.0, 2047: 2.0, 2048: 2.0, 2049: 2.0, 2050: 2.0, 2051: 2.0, 2052: 2.0, 2053: 2.0, 2054: 2.0, 2055: 2.0, 2056: 2.0, 2057: 2.0, 2058: 2.0, 2059: 2.0, 2060: 2.0, 2061: 2.0, 2062: 2.0}


In [ ]:
df_technology_month_currency_transformed['year'].unique()
df_system_month_currency_transformed['year'].unique()


array([2022, 2023, 2024, 2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032,
       2033, 2034, 2035, 2036, 2037, 2038, 2039, 2040, 2041, 2042, 2043,
       2044, 2045, 2046, 2047, 2048, 2049, 2050, 2051, 2052, 2053, 2054,
       2055, 2056, 2057, 2058, 2059, 2060])

In [ ]:
df_technology_month_currency_transformed.rename(columns={'Weighted capture price': 'value',
                                                         'nominal_Weighted capture price': 'nominal_value',
                                                         'Group' : 'variable'}, inplace=True)

df_concat = pd.concat([df_system_month_currency_transformed, df_technology_month_currency_transformed], ignore_index=True)
df_concat.to_pickle('c:/Users/mpy/Market data retrieval/api_retrieval/data/aurora/processed/aurora_prices_ES_default_currency_1m_nominal_prices.pkl')
df_concat
df_concat['year'].unique()

array([2022, 2023, 2024, 2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032,
       2033, 2034, 2035, 2036, 2037, 2038, 2039, 2040, 2041, 2042, 2043,
       2044, 2045, 2046, 2047, 2048, 2049, 2050, 2051, 2052, 2053, 2054,
       2055, 2056, 2057, 2058, 2059, 2060])

In [ ]:
df_concat.to_csv('c:/Users/mpy/Market data retrieval/api_retrieval/data/aurora/processed/aurora_prices_ES_default_currency_1m_nominal_prices.csv')